<a href="https://colab.research.google.com/github/effat/SQA-2026/blob/main/SQA-2026/tree/main/Workshops/Workshop-5/In_Class_Demo_Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ================================
# 1. Install dependencies
# ================================

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip -q install --upgrade transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


# ================================
# 2. Import libraries
# ================================

In [3]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ================================
# 3. System Prompt: Zero Shot Setting
# ================================

In [4]:
SYSTEM_PROMPT = """
You are a Senior Software Requirements Engineer working in a regulated compliance environment.

Your task is to convert regulatory legal text into precise, testable, system-level software requirements.

Rules:
1. Only extract explicit regulatory obligations.
2. Do NOT hallucinate new rules.
3. Do NOT interpret beyond the text.
4. Each requirement must begin with: "System shall".
5. Requirements must be atomic (one obligation per requirement).
6. Output MUST be valid JSON.
7. Do not include explanations.
8. Do not merge separate obligations.
9. Preserve time constraints exactly as written.
10. Identify role-based obligations explicitly (e.g., PCQI).
"""

# ================================
# 4. Prompt Builder
# ================================

In [5]:
def build_prompt(reg_text):
    return f"""
[INST] <<SYS>>
{SYSTEM_PROMPT}
<</SYS>>
Convert the following regulatory text into atomic system requirements in JSON ONLY:

{reg_text}
[/INST]
"""

# ================================
# 5. Load Model
# ================================

In [6]:
#Make sure the model is loaded on GPU
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

# ================================
# 6. Generation Function
# ================================

In [7]:
def generate(model, text, max_new_tokens=200):
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# ================================
# 7. Regulatory Text Example
# ================================
From 21 CFR 117.130 under U.S. Food and Drug Administration.
https://www.ecfr.gov/current/title-21/chapter-I/subchapter-B/part-117/subpart-C/section-117.130

In [9]:
reg_text = """
The owner, operator, or agent in charge of a facility must conduct a hazard analysis
to identify and evaluate known or reasonably foreseeable hazards for each type of
food manufactured, processed, packed, or held at the facility.

The hazard analysis must be written.
"""

# ================================
# 8. Run Model
# ================================

In [10]:
output_fp16 = generate(model_fp16, reg_text)
print("===== FP16 MODEL OUTPUT =====")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


===== FP16 MODEL OUTPUT =====


Print Output

In [11]:
print(output_fp16)


[INST] <<SYS>>

You are a Senior Software Requirements Engineer working in a regulated compliance environment.

Your task is to convert regulatory legal text into precise, testable, system-level software requirements.

Rules:
1. Only extract explicit regulatory obligations.
2. Do NOT hallucinate new rules.
3. Do NOT interpret beyond the text.
4. Each requirement must begin with: "System shall".
5. Requirements must be atomic (one obligation per requirement).
6. Output MUST be valid JSON.
7. Do not include explanations.
8. Do not merge separate obligations.
9. Preserve time constraints exactly as written.
10. Identify role-based obligations explicitly (e.g., PCQI).

<</SYS>>
Convert the following regulatory text into atomic system requirements in JSON ONLY:


The owner, operator, or agent in charge of a facility must conduct a hazard analysis
to identify and evaluate known or reasonably foreseeable hazards for each type of
food manufactured, processed, packed, or held at the facility.


# ================================
# 9. Parse Output
# ================================

In [12]:
def extract_requirements(output_text):
    try:
        # find first { in the text
        json_start = output_text.find("{")
        parsed = json.loads(output_text[json_start:])
        return parsed.get("requirements", [])
    except Exception as e:
        print("Error parsing JSON:", e)
        return []

# ================================
# 10. Ground Truth
# ================================

In [ ]:
ground_truth = [
"System shall require the owner, operator, or agent in charge of the facility to conduct a hazard analysis.",
"System shall identify known hazards for each type of food manufactured, processed, packed, or held at the facility.",
"System shall identify reasonably foreseeable hazards for each type of food manufactured, processed, packed, or held at the facility.",
"System shall ensure the hazard analysis is documented in written form."
]

In [ ]:
# ================================
#10. Assess Model Output
# ================================

We check which predicted requirements are correct (true positives):

"System shall store and retrieve written hazard analysis..."

Ground truth has: "System shall ensure the hazard analysis is documented in written form."

 This is similar, but not exactly the same obligation — model added "store and retrieve", which is extra.

Count as false positive (not a match).

"System shall conduct hazard analysis for identification and evaluation of known or reasonably foreseeable hazards..."

Matches ground truth requirement #1 and #2 combined: "System shall require the owner, operator, or agent in charge of the facility to conduct a hazard analysis." and "System shall identify known hazards..."

The model merged obligations (violates atomicity rule).

For strict evaluation, merged obligations are counted as 1 true positive (we could optionally count 0.5 per obligation, but standard is 1).

True positives: 1
 False positives: 1

In [13]:
# Ground truth requirements
ground_truth = [
    "System shall require the owner, operator, or agent in charge of the facility to conduct a hazard analysis.",
    "System shall identify known hazards for each type of food manufactured, processed, packed, or held at the facility.",
    "System shall identify reasonably foreseeable hazards for each type of food manufactured, processed, packed, or held at the facility.",
    "System shall ensure the hazard analysis is documented in written form."
]

# Predicted requirements from model
predicted = [
    "System shall store and retrieve written hazard analysis for each type of food manufactured, processed, packed or held at the facility.",
    "System shall conduct hazard analysis for identification and evaluation of known or reasonably foreseeable hazards for each type of food manufactured, processed, packed or held at the facility."
]


true_positives = 1
'''
def evaluate_fuzzy(predicted, ground_truth):
    true_positives = 0
    used_gt = set()  # keep track of matched ground truth
    for p in predicted:
        for i, gt in enumerate(ground_truth):
            # consider a match if the predicted text contains key phrases from ground truth
            if i not in used_gt and "hazard analysis" in p.lower() and "food" in p.lower():
                true_positives += 1
                used_gt.add(i)
                break
'''

precision = true_positives / len(predicted) if predicted else 0
recall = true_positives / len(ground_truth) if ground_truth else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
  #return precision, recall, f1

#precision, recall, f1 = evaluate_fuzzy(predicted, ground_truth)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1:.2f}")



Precision: 0.50
Recall: 0.25
F1-score: 0.33
